# 06 - Synthetic Collaborative Filtering Interactions Analysis

This notebook profiles `fitgenius_cf_synthetic_interactions.csv.gz` and shows how the backend converts interaction rows into collaborative-filtering priors for exercises and meals.

The recommender uses this data as a cold-start prior only. Real app feedback keeps a higher weight.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path('../data/fitgenius_cf_synthetic_interactions.csv.gz')
if not DATA_PATH.exists():
    DATA_PATH = Path(r'C:/Users/91700/Downloads/fitgenius_cf_synthetic_interactions.csv.gz')

DATA_PATH

In [ ]:
df = pd.read_csv(DATA_PATH, compression='gzip')
df.shape, df.head()

## Schema and Missingness

In [ ]:
schema = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(df[col].dtype) for col in df.columns],
    'missing_pct': [round(df[col].isna().mean() * 100, 2) for col in df.columns],
    'unique_values': [df[col].nunique(dropna=True) for col in df.columns],
})
schema

## Distribution Checks

In [ ]:
summary_tables = {
    'item_type': df['item_type'].value_counts(dropna=False),
    'fitness_goal': df['fitness_goal'].value_counts(dropna=False),
    'diet_preference': df['diet_preference'].value_counts(dropna=False),
    'experience_level': df['experience_level'].value_counts(dropna=False),
    'outcome_label': df['outcome_label'].value_counts(dropna=False),
    'safety_violation_flag': df['safety_violation_flag'].value_counts(dropna=False),
}
summary_tables

## Backend-Compatible Score Formula

This mirrors `Backend/recommendations/collaborative.py`.

- Positive signals: implicit score, rating, completed, started/eaten, final rank score
- Negative signals: skipped, pain reported, safety violation
- Scores are clipped to `[-8, 8]` and averaged per item within the matched cohort

In [ ]:
def bool_col(series):
    return series.astype(str).str.lower().isin(['1', 'true', 'yes', 'y'])

work = df.copy()
for col in ['implicit_feedback_score', 'rating_1_5', 'final_rank_score']:
    work[col] = pd.to_numeric(work[col], errors='coerce').fillna(0)

work['backend_cf_score'] = (
    work['implicit_feedback_score']
    + work['rating_1_5']
    + work['final_rank_score']
    + np.where(bool_col(work['completed']), 2.0, 0.0)
    + np.where(bool_col(work['started_or_eaten']), 1.0, 0.0)
    - np.where(bool_col(work['skipped']), 2.0, 0.0)
    - np.where(bool_col(work['pain_reported']), 5.0, 0.0)
    - np.where(bool_col(work['safety_violation_flag']), 8.0, 0.0)
).clip(-8, 8)

work['backend_cf_score'].describe()

## Top and Bottom Items

In [ ]:
item_scores = (
    work[work['item_type'].isin(['exercise', 'meal'])]
    .groupby(['item_type', 'item_name'])
    .agg(avg_score=('backend_cf_score', 'mean'), interactions=('interaction_id', 'count'), safety_violations=('safety_violation_flag', lambda s: bool_col(s).sum()))
    .query('interactions >= 50')
    .sort_values('avg_score', ascending=False)
)

display(item_scores.head(20))
display(item_scores.tail(20))

## Example Cohort Match

This approximates a FitGenius profile and shows the items the backend would treat as useful priors.

In [ ]:
profile = {
    'fitness_goal': 'fat_loss',
    'gender': 'male',
    'experience_level': 'beginner',
    'diet_preference': 'vegetarian',
}

cohort = work[
    (work['fitness_goal'] == profile['fitness_goal'])
    & (work['gender'] == profile['gender'])
    & (work['experience_level'] == profile['experience_level'])
    & (work['diet_preference'] == profile['diet_preference'])
    & (work['item_type'].isin(['exercise', 'meal']))
]

cohort_scores = (
    cohort.groupby(['item_type', 'item_name'])
    .agg(avg_score=('backend_cf_score', 'mean'), interactions=('interaction_id', 'count'))
    .query('interactions >= 10')
    .sort_values('avg_score', ascending=False)
)

cohort.shape, cohort_scores.head(20)